# データ・AI活用実践（初級）第04回

-------------------

## 自分で作った「扱いやすい」データファイルの読み込み

本日も今までと同様，全国の気温データを扱います．しかし，毎回毎回あの過酷な前処理をやるのもあほらしいですし，今回はもっと別のことに時間を使いたいです．そこで，第３回の成果である，**変なヘッダや謎の文字列を削除して，実数データと時刻データだけにした扱いやすいデータファイル**を使いましょう．

このページの左にある「フォルダマーク」をクリックすると，「ファイル」というサイドバーが出てくると思います．`sample_data`というフォルダがすでにあって，その下には結構広いスペースがあると思います．そこに，
第3回の自由課題で自分のPCにダウンロードした`pref47_temp.csv`というファイルを，ドラッグ＆ドロップすることでアップロードしてください．

「いや，もうわけわからん」という人は，仕方ないので以下を実行してみてください．

In [ ]:
# ここは覚えなくても大丈夫！
csvurl <- "https://www.cc.kyoto-su.ac.jp/~ogohara/lecture/DataAI_FirstCourse/pref47_temp.csv"
download.file(csvurl, destfile = "pref47_temp.csv") #pref47_temp.csvという名前のファイルとしてダウンロード

そして，いつもそうですが，ライブラリを使えるようにしましょう．ただし，今回はインストールされていないライブラリを使用します．新たにインストールする必要があるのですが，注意点は以下の通りです．

- インストールするのに数十分かかるかもしれない
- ランタイムを再起動したり，Colabを一度終わってからもう一度始めたりすると，一からインストールしなおしになる

したがって，小郷原が解説をし始める**まさに今かから以下のコードを実行して，解説を聞いている間にインストールが完了するように**しましょう．小郷原の場合は20分以上かかりました．

In [ ]:
system("apt-get install -y libudunits2-dev libgdal-dev libgeos-dev libproj-dev") #もしかしたらいらないかもしれない
install.packages('sf')

In [ ]:
install.packages('NipponMap')

In [ ]:
system('apt-get install libprotobuf-dev protobuf-compiler')
system('apt-get install libjq-dev')
install.packages('geojsonio')

In [ ]:
library(tidyverse)
library(sf)
library(NipponMap)

次は，本命のデータファイルを読み込みます．

In [ ]:
df<-read_csv('pref47_temp.csv', locale=locale(encoding='Shift_JIS'))

中身を確認しましょう．

In [ ]:
df

余計な1列が一番左に入っているかもしれません．まああってもいいのですが，せっかく苦労して行った前処理が無駄になっている気がして癪なので消してしましましょう．もちろん，csvファイルを保存するときに，こういう無駄な列が入らないように保存する方法はありますので，我慢ならない人は調べてみてください．

In [ ]:
# 整数列からなる余計な1列が左端に入っていない人は
# df47 <- df
# としてください．
df47 <- df[,-1]

--------------------
## 一気に全都道府県比較

前回はせいぜい近畿だけとか，那覇と福井と札幌だけ，とか，ごく限られた都道府県間の比較でした．それでも十分楽しいのですが，今回はとにかく全部グラフに書いてみて，迫力のある演習にしましょう．

In [ ]:
summary(df47)

当たり前ですけど，こんなにたくさんの数字を出されてもパッと見てわかりません．

In [ ]:
mtsum <- apply(df47[,-1], 2, summary)
mtsum

これ，何が変わったのかわかります？？データフレームっぽくなったのです．まあデータフレームではないのですが，まあいいでしょう．ちゃんとデータフレームに変換してから使います．`date`列が消えてしまいましたが，ここでは時系列データを比較するわけではないので，これでいいことにしましょう．

In [ ]:
dfsum <- data.frame(t(mtsum))
dfsum

はい，データフレームに一発変換できました．見やすいですね．例えば`Mean`列を見れば，すべての都道府県の**全期間平均値**がわかります．

データフレームにしたのですから，もちろんここから棒グラフなどを作成したってかまいません．データフレームにする前の状態`mtsum`から直接描画することもできます．いろんな方法があるといことを勉強する意味で，あえてデータフレーム`dfsum`を使わずに棒グラフを書いてみましょう．

In [ ]:
mtsum["Mean",]

このようにmatrixのある行をスライスすると，列名とともに表示されます．この場合，`barplot`という関数を使うことでサクッと棒グラフを作成できるのです．

In [ ]:
#何もしないと日本語が文字化けします．日本語フォントをインストールします．
system("apt-get install -y fonts-noto-cjk")
#Noto Sans CJK JPというフォントを使います宣言
theme_set(theme_bw(base_family = "Noto Sans CJK JP"))

In [ ]:
barplot(mtsum["Mean",],las=2,xlab='Prefecture', ylab='Mean temperature [deg_C]')

人によっては文字化けしていると思います．気持ち悪いことには同意しますが，文字化けを修復していると授業が進まないので，とりあえず放置することにします．

`las=2`は横軸のラベルを回転させています．`xlab`, `ylab`は横軸縦軸の軸名です．おそらく横軸の都道府県名はスペースの都合で一部省略されているでしょう．

ざっくり右肩下がりといいますか，なんとなく九州・沖縄は気温が高く，東北以北は寒い気がしませんか？細かな極小は，山陰や信越に対応しています．

---------------
## 地図上に描画する

都道府県別の平均気温を求められたら，日本地図の上に表示してみたくなりませんか？暖かい自治体は赤で，寒い自治体は青で塗りつぶして，日本のどのあたりが温かいは一目見てわかるようなずです．そのような図は，数値のられるを見せられるよりも，ヒストグラムで見せられるよりも，ずっと多くのことをずっと短い時間で読者に伝えることができます．ただし，それをRでやろうとすると結構骨が折れます．ここではまずはRを使って日本地図を描くところから勉強しましょう．

### 日本地図

日本地図を描くには何が必要か？地図データです．皆さんのColabには入っていないと思いますが，本日の最初のほうですでにインストールできているはずです．順番にセルを実行しているならば，すでに`library(なんたら)`も実行できているはずですね．

とりあえず`NipponMap`といういかにもな名前のライブラリに含まれている，日本の形状ファイル（シェイプファイル）を読み込みましょう．

In [ ]:
map <- read_sf(system.file("shapes/jpn.shp", package = "NipponMap")[1],
                crs = "+proj=longlat +datum=WGS84")

`crs`オプションって何？と思うかもしれませんが，まあ細かいことは置いておいて，地図投影法くらいに思っておいてください．`map`の中身を確認します．

> もしかしたらエラーになるかもしれません．しかし，最後にデータフレームらしきものが出力されていれば問題ないです．

In [ ]:
map

`name`列は皆さんがよく知っている都道府県名です．ご丁寧に人口データ`population`も含まれていますね．`geometry`列が都道府県の形を表しているのですが，中身を見れるわけではないのでスルーでいいです．`jiscode`が都道府県コードです．今ある人口データを簡単に描画しようと思えば，想像以上に簡単に描画できます．

In [ ]:
ggplot(map, aes(fill = population)) + geom_sf() + labs(title = "Population")

おおーーすげーー．日本地図なので，当然ながら朝鮮半島は含まれていません．琵琶湖や霞ケ浦，佐渡島，淡路島もありません．私は滋賀県に住んでいるのですが，琵琶湖がないのは度し難いです．まあ描こうと思えば描けるので今回は我慢しましょう（チッ）．こういう図を平均気温で作りたいです．

--------------------
### 都道府県名が一致しない問題

さて，ここで１つ大問題があります．`dfsum`に入っている都道府県名と`map`に入っている`name`列が一致しません．日本語とアルファベットの差もありますが，そもそも「つくば（館野）」なんていう都道府県はありません．`dfsum`の平均気温データをしかるべき`name`に対応させる必要があります．今回は，「熊谷は埼玉県だからSaitamaで．．．」などと考えるのではなくて，`jiscode`列を使うのがいいでしょう．つまり，`dfsum`に新しく`jiscode`列を作れれば，`jiscode`を通して，平均気温を`map`に追加することができるかもしれません．じゃあどうするのか？とてもエレガントな方法があるのかもしれませんが，おそらく今回は泥臭くやったほうが早いです．まずは中身を確認しましょう．

In [ ]:
rownames(dfsum) #行(row)の名前(name)

次に，比較のためにmapの中身を表示します．

In [ ]:
map$name #mapの中のname列

どうやら，南北逆順に並んでいるようです．ただし，厳密に逆順なのわけでもないです．したがって，もう一つ一つ対応付けるほかないですね．やりたいことは以下の通りです．ゆくゆくはこれを自分で考えられるようになりたいですね．

1. `dfsum`内の都道府県名（都市名含む）を`JISCODE`に対応付ける
1. `JISCODE N` に対応する都道府県の平均気温を，`map`内の当該`JISCODE N`の行の新たな列`temperature`に転記する
1. `map`を使って地図上に平均気温をプロットする

各都道府県には番号が振られています．各都道府県の中にある各市町村にも番号が振られていて，住所を何桁かの整数で表すことができるようになっているのです．それはJISによって定められていて，検索するといろんなところから入手することが可能です．例えばここを見れば，[ここ](https://nlftp.mlit.go.jp/ksj/gml/codelist/PrefCd.html "都道府県コード")を見れば一発でわかります．今回はすでに私が準備していますので，以下のコードを使ってダウンロードしてください．

In [ ]:
# ここは覚えなくても大丈夫！
csvurl <- "https://www.cc.kyoto-su.ac.jp/~ogohara/lecture/DataAI_FirstCourse/JIS.csv"
download.file(csvurl, destfile = "JIS.csv") #data.csvという名前のファイルとしてダウンロード

JISコード表を読み込んでみます．

In [ ]:
jisdf <- read_csv('JIS.csv', locale=locale(encoding='Shift_JIS'))

In [ ]:
jisdf

自分の出身地のJISコードって知ってました？？

中身を見ていただければわかりますが，都道府県名列には「県」とか「府」が付いていて，`dfsum`や`map`の中の都道府県表記とは異なります．北海道だけは余計なものが付いていません．これ自体はまあ何とかなります．面倒なのは，`dfsum`で都市名を都道府県名に変換しないといけないことです．

In [ ]:
city <- c('那覇', '松江', '松山', '高松', '神戸', '津', '彦根', '金沢', '名古屋', '前橋', '甲府', '横浜', '熊谷', '宇都宮', 'つくば（館野）', '仙台', '盛岡', '札幌')
pref <- c('沖縄', '島根', '愛媛', '香川', '兵庫', '三重', '滋賀', '石川', '愛知', '群馬', '山梨', '神奈川', '埼玉', '栃木', '茨城', '宮城', '岩手', '北海道')

しんどいですね．．．．まあ1回やればいいわけですから我慢しましょう．本来なら，皆さん自身が自分の手で書く必要があります．この辺りで私がどういう順番で何をしようとしているかわかっていただけるでしょうか？最初にやりたいことは，`dfsum`の中の都道府県名（都市名含む）と`JISCODE`を対応付けることです．`jisdf`の都道府県名を上から1つずつチェックして，`jisdf`の都道府県名から最後の1文字を削除した残り（北海道除く）と`dfsum`の都道府県名（都市名含む）と一致すれば，`jisdf`の当該行の第1列の整数を`dfsum`の`jiscode`列に追加します．

In [ ]:
for (n in jisdf$都道府県名){ #jisdfの都道府県名列の値を「順番」にnに代入
  nlen <- str_length(n)      #nに代入された文字列の長さ
  if (n=="北海道") {         #nが「北海道」だったら
    prefname <- n            #prefnameにnの文字列を代入
  } else {                   #そうでなかったら
    prefname <- str_sub(n, end=nlen-1) #nの文字列のうち，最初からnlen-1番目までを抜き出してprefnameに代入
  }

  # 以下のソースコードを解読してみましょう．
  if ( is.na(match(prefname,pref)) ) {
    name_in_dfsum <- prefname
  } else {
    name_in_dfsum <- city[which(pref == prefname)]
  }
  dfsum[name_in_dfsum,'jiscode'] <- jisdf[ which(jisdf$都道府県名 == n),"都道府県コード" ]
}

試しに，`dfsum`の中身を確認してみてください．ちゃんと`jiscode`列が追加されていますか？このソースコードの各行が一体何をしているのか，必ず理解してから進んでください．泥臭くてもいいので，とにかくやりたいことを実現できるソースコードをひねり出せることが，成長の第1歩です．

> ColabにはAIがソースコードを解説してくれる機能があります．ぜひ使ってみてください．ただし，英語でしか解説してくれないかもしれません．

`map`の中のデータはどうやら`jiscode`の順に上から並んでいるようなので，`dfsum`も`jiscode`の順に並び変えれば勝手に都道府県が対応してくれるはずです．

In [ ]:
dfsum[order(dfsum$jiscode),]

どうですか？jiscode列の値の順に並んでますか？

では，ようやくです．ようやく`map`に新しい列を追加できます．

In [ ]:
map[,'temperature'] <- dfsum[order(dfsum$jiscode),'Mean']
map

よし！できました！

----------------
### 平均気温を日本地図上にプロットする

いよいよ最後の作品です．といっても，残っていることはまさに図を書くことだけですから，今までのコピペで済んでしまいます．

In [ ]:
library(RColorBrewer)

In [ ]:
ggplot(map, aes(fill = temperature)) + geom_sf() + labs(title = "Mean temperature") + scale_fill_gradientn(colours=topo.colors(9))

長かった．．．．．長野県が涼しげですね．

## 自由課題


1. データの期間内における最高気温を都道府県別に日本地図上に表示してください．
2. データの期間内における最高気温と最低気温の差を都道府県別に日本地図上に表示してください．
